# AksharDrishti: synthetic benchmark evaluation
Use **Runtime > Run all**. Cell 2 installs and restarts once; then **Run all** again. Needs a T4 GPU and the `HF_TOKEN` secret.

In [ ]:
# 1. Check the GPU
!nvidia-smi

In [ ]:
# 2. Install everything once, then restart automatically (Run all again after the restart)
import os
if not os.path.exists("/content/.deps_ok"):
    !pip install -q -U "transformers>=5.7" "accelerate>=1.1" "huggingface_hub>=1.0" "pillow>=10.0.1,<13" ultralytics gradio openpyxl lxml
    !apt-get -qq install -y fonts-noto-core > /dev/null
    open("/content/.deps_ok", "w").write("ok")
    print("Installed. Restarting the session - then use Runtime > Run all again.")
    os.kill(os.getpid(), 9)
print("dependencies already installed")

In [ ]:
# 3. Log in to Hugging Face and download the model + weights (~1.8 GB)
from google.colab import userdata
from huggingface_hub import login, snapshot_download

login(userdata.get("HF_TOKEN"))
repo = snapshot_download("bodhan-ai/indic-ocr", local_dir="/content/indic-ocr")
print("Downloaded to", repo)

In [ ]:
# 4. Load the model (once per session)
import sys, torch
sys.path.insert(0, repo)
from indic_ocr import IndicOCR

print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
parser = IndicOCR.from_pretrained(repo)   # add table_format="markdown" for Markdown tables

In [ ]:
# S1. Install YOLO and load the detectors
# (ultralytics was installed in cell 2)
from huggingface_hub import hf_hub_download
from ultralytics import YOLO

sig_model = YOLO(hf_hub_download("tech4humans/yolov8s-signature-detector", "yolov8s.pt"))

STAMP_WEIGHTS = None   # e.g. "/content/drive/MyDrive/stamp_best.pt" once you have a trained stamp model
stamp_model = YOLO(STAMP_WEIGHTS) if STAMP_WEIGHTS else None
print("signature classes:", sig_model.names, "| stamp model:", STAMP_WEIGHTS or "colour detector")

In [ ]:
# S2. Detection + merge helpers
import cv2, numpy as np, os, json

def covered(a, b):
    """Fraction of box a that lies inside box b."""
    x0, y0, x1, y1 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x1 - x0) * max(0, y1 - y0)
    area = (a[2] - a[0]) * (a[3] - a[1])
    return inter / area if area else 0.0

def yolo_boxes(model, path, kind, conf):
    r = model(path, conf=conf, imgsz=1024, verbose=False)[0]
    return [{"type": kind, "bbox_xyxy": [round(v, 1) for v in b], "conf": round(c, 3), "method": "yolo"}
            for b, c in zip(r.boxes.xyxy.tolist(), r.boxes.conf.tolist())]

def colour_stamps(path, min_area_frac=0.002):
    """Stamps are usually coloured ink: find large saturated blobs."""
    img = cv2.imread(path)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, (0, 70, 50), (180, 255, 255))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((25, 25), np.uint8))
    n, _, stats, _ = cv2.connectedComponentsWithStats(mask)
    H, W = mask.shape
    out = []
    for x, y, w, h, a in stats[1:]:
        if a < min_area_frac * H * W or not (0.4 < w / h < 2.5):
            continue
        out.append({"type": "stamp", "bbox_xyxy": [float(x), float(y), float(x + w), float(y + h)],
                    "conf": None, "method": "colour"})
    return out

def detect_marks(path, sig_conf=0.4, stamp_conf=0.4):
    sigs = yolo_boxes(sig_model, path, "signature", sig_conf)
    if stamp_model:
        stamps = yolo_boxes(stamp_model, path, "stamp", stamp_conf)
    else:
        # blue-ink signatures also look like colour blobs: drop blobs that are mostly a signature
        stamps = [s for s in colour_stamps(path)
                  if not any(covered(s["bbox_xyxy"], g["bbox_xyxy"]) > 0.6 for g in sigs)]
    return sigs + stamps

def add_marks(path, page, crop_dir="/content/output/crops"):
    from PIL import Image
    os.makedirs(crop_dir, exist_ok=True)
    im = Image.open(path).convert("RGB")
    marks = detect_marks(path)
    name = os.path.splitext(os.path.basename(path))[0]
    for i, m in enumerate(marks):
        m["crop"] = f"{crop_dir}/{name}_{m['type']}_{i}.png"
        im.crop(m["bbox_xyxy"]).save(m["crop"])
    # tag IndicOCR blocks that are really a stamp/signature (e.g. a seal read as a paragraph)
    for b in page["blocks"]:
        for m in marks:
            if covered(b["bbox_xyxy"], m["bbox_xyxy"]) > 0.5:
                b["inside_mark"] = m["type"]
    page["marks"] = marks
    return page

In [ ]:
# A2. Key-field extraction + privacy masking
import re, io, os, html, json, base64, sqlite3, datetime, unicodedata
import pandas as pd
from PIL import Image, ImageDraw

DEV_DIGITS = str.maketrans("०१२३४५६७८९", "0123456789")   # Devanagari -> ASCII digits (1:1, so spans line up)

def clean(s):
    return (s or "").replace("▁", " ")

def to_plain(text):
    """Block text -> plain text (tables come back as HTML)."""
    t = clean(text)
    t = re.sub(r"<br\s*/?>", " ", t)
    t = re.sub(r"</t[dh]>", " ", t)
    t = re.sub(r"</tr>", "\n", t)
    t = re.sub(r"<[^>]+>", "", t)
    return html.unescape(t).replace("$", "").strip()

FIELD_PATTERNS = {
    "Aadhaar":          (r"(?<!\d)[2-9]\d{3}\s?\d{4}\s?\d{4}(?!\d)", 0),
    "PAN":              (r"(?<![A-Z0-9])[A-Z]{5}\d{4}[A-Z](?![A-Z0-9])", 0),
    "Phone":            (r"(?<!\d)(?:\+91[\s-]?)?[6-9]\d{4}\s?\d{5}(?!\d)", 0),
    "PIN code":         (r"(?<!\d)[1-9]\d{2}\s?\d{3}(?!\d)", 0),
    "Date":             (r"(?<!\d)\d{1,2}[./-]\d{1,2}[./-](?:\d{4}|\d{2})(?!\d)", 0),
    "Amount":           (r"(?:₹|rs\.?|रु\.?|रुपये)\s?\d[\d,]*(?:\.\d{1,2})?", re.I),
    "Survey / Gat no.": (r"(?:survey|sy\.|gat|गट|सर्वे|सर्व्हे)\s*(?:no\.?|number|नं\.?|क्र\.?|क्रमांक)?\s*[:.\-]?\s*\d+(?:/\d+)*", re.I),
    "Email":            (r"[\w.+-]+@[\w-]+\.[\w.]+", 0),
}
SENSITIVE = ("Aadhaar", "PAN")

def find_spans(text, kinds=None):
    """[(kind, start, end)] found in text (Devanagari digits included)."""
    t = (text or "").translate(DEV_DIGITS)
    out = []
    for kind, (pat, flags) in FIELD_PATTERNS.items():
        if kinds and kind not in kinds:
            continue
        out += [(kind, m.start(), m.end()) for m in re.finditer(pat, t, flags)]
    return out

def extract_fields(text):
    t = (text or "").translate(DEV_DIGITS)
    rows, seen = [], set()
    for kind, s, e in find_spans(text):
        v = " ".join(t[s:e].split())
        if (kind, v) not in seen:
            seen.add((kind, v)); rows.append([kind, v])
    return rows

def mask_value(kind, v):
    d = re.sub(r"\s", "", v.translate(DEV_DIGITS))
    return "XXXX XXXX " + d[-4:] if kind == "Aadhaar" else "XXXXXX" + d[-4:]

def mask_text(text):
    if not text:
        return text
    for kind, s, e in sorted(find_spans(text, SENSITIVE), key=lambda x: -x[1]):
        text = text[:s] + mask_value(kind, text[s:e]) + text[e:]
    return text

def sensitive_boxes(page):
    """Approximate image boxes of Aadhaar/PAN: the matching line band inside the block."""
    boxes = []
    for b in page["blocks"]:
        lines = to_plain(b.get("text")).split("\n")
        x0, y0, x1, y1 = b["bbox_xyxy"]
        lh = (y1 - y0) / max(len(lines), 1)
        for i, line in enumerate(lines):
            for _, s, e in find_spans(line, SENSITIVE):
                n = max(len(line), 1)
                bx0 = x0 + (x1 - x0) * max(s / n - 0.05, 0)
                bx1 = x0 + (x1 - x0) * min(e / n + 0.05, 1)
                boxes.append([bx0, y0 + i * lh, bx1, y0 + (i + 1) * lh])
    return boxes

# quick self-test
_t = "आधार क्र. २३४५ ६७८९ ०१२३, PAN ABCDE1234F, mob 9876543210, dt 12/03/2024, गट नं. 45/2, ₹ 1,500"
print(extract_fields(_t)); print(mask_text(_t))

In [ ]:
# A3. Searchable archive (SQLite) - every processed document is stored here
DB = "/content/archive.db"          # change to a Drive path to keep it between sessions
con = sqlite3.connect(DB, check_same_thread=False)
con.execute("""CREATE TABLE IF NOT EXISTS docs(
    id INTEGER PRIMARY KEY, name TEXT, added TEXT,
    signed INT, stamped INT, fields TEXT, text TEXT)""")
con.commit()

def nfc(s):
    return unicodedata.normalize("NFC", s or "")

def archive_add(name, signed, stamped, fields, text):
    con.execute("INSERT INTO docs(name, added, signed, stamped, fields, text) VALUES (?,?,?,?,?,?)",
                (name, datetime.datetime.now().strftime("%Y-%m-%d %H:%M"), int(signed), int(stamped),
                 json.dumps(fields, ensure_ascii=False), nfc(text)))
    con.commit()

def archive_search(q):
    q = nfc(q).strip()
    rows = con.execute("SELECT id, name, added, signed, stamped, text FROM docs WHERE text LIKE ? OR name LIKE ? "
                       "ORDER BY id DESC", (f"%{q}%", f"%{q}%")).fetchall()
    out = []
    for i, name, added, sg, st, text in rows:
        k = text.find(q) if q else 0
        snip = text[max(k - 60, 0): k + len(q) + 60].replace("\n", " ") if k >= 0 else text[:120]
        out.append([i, name, added, "✔" if sg else "✘", "✔" if st else "✘", "…" + snip + "…"])
    return pd.DataFrame(out, columns=["ID", "Document", "Added", "Signed", "Stamped", "Match"])

In [ ]:
# A4. Full pipeline: OCR -> stamps/signatures -> fields -> masking -> exports -> archive
BLOCK_COLORS = {"Table": (22, 163, 74), "Title": (249, 115, 22), "SectionHeader": (249, 115, 22)}
MARK_COLORS = {"signature": (220, 38, 38), "stamp": (124, 58, 237)}
MAX_SIDE = 2000
import gc, torch, time

def report_html(name, verdict, fields, page, img_path):
    b64 = base64.b64encode(open(img_path, "rb").read()).decode()
    rows = "".join(f"<tr><td>{html.escape(k)}</td><td>{html.escape(v)}</td></tr>" for k, v in fields)
    body = ""
    for b in sorted(page["blocks"], key=lambda b: b["order"]):
        t = clean(b.get("text"))
        if not t:
            continue
        body += t if b.get("type") == "Table" else f"<p>{html.escape(t)}</p>"
    return f"""<!DOCTYPE html><html><head><meta charset="utf-8"><title>{html.escape(name)}</title>
<style>body{{font-family:'Noto Sans','Noto Sans Devanagari',sans-serif;max-width:900px;margin:auto;padding:20px}}
table{{border-collapse:collapse;margin:10px 0}}td,th{{border:1px solid #999;padding:4px 8px}}img{{max-width:100%}}</style>
</head><body><h1>Document report: {html.escape(name)}</h1>
<p>Generated {datetime.datetime.now():%Y-%m-%d %H:%M} by AksharDrishti (OCR engine: IndicOCR by Bodhan AI)</p>
<h2>Verification</h2><pre>{html.escape(verdict)}</pre>
<h2>Key fields</h2><table><tr><th>Field</th><th>Value</th></tr>{rows}</table>
<h2>Annotated page</h2><img src="data:image/png;base64,{b64}">
<h2>Extracted text</h2>{body}</body></html>"""

def process_document(path, mask=True):
    if not path:
        raise gr.Error("Upload a page image first.")
    name = os.path.splitext(os.path.basename(path))[0]
    out = f"/content/app_output/{name}"
    os.makedirs(out, exist_ok=True)

    # 0. shrink big scans/photos (they crash Colab's RAM); 2000 px long side is plenty for OCR
    im0 = Image.open(path).convert("RGB")
    if max(im0.size) > MAX_SIDE:
        im0.thumbnail((MAX_SIDE, MAX_SIDE))
    path = f"{out}/{name}_input.png"
    im0.save(path)
    del im0

    # 1. OCR (IndicOCR)
    t0 = time.time()
    page = parser.parse(path)
    t1 = time.time()
    page["markdown"] = clean(page["markdown"])
    for b in page["blocks"]:
        b["text"] = clean(b.get("text"))

    # 2. stamps + signatures (cells S1/S2)
    page = add_marks(path, page, crop_dir=f"{out}/crops")
    t2 = time.time()
    sigs = [m for m in page["marks"] if m["type"] == "signature"]
    stamps = [m for m in page["marks"] if m["type"] == "stamp"]

    # 3. key fields
    full = "\n".join(to_plain(b["text"]) for b in sorted(page["blocks"], key=lambda b: b["order"]) if b["text"])
    fields = extract_fields(full)

    # 4. privacy masking
    hide = sensitive_boxes(page) if mask else []
    if mask:
        fields = [[k, mask_value(k, v) if k in SENSITIVE else v] for k, v in fields]
        for b in page["blocks"]:
            b["text"] = mask_text(b["text"])
        page["markdown"] = mask_text(page["markdown"])
        full = mask_text(full)
        for m in page["marks"]:   # crops could show a number next to a signature
            im_c = Image.open(m["crop"])
            dx, dy = m["bbox_xyxy"][0], m["bbox_xyxy"][1]
            for hb in hide:
                ImageDraw.Draw(im_c).rectangle([hb[0] - dx, hb[1] - dy, hb[2] - dx, hb[3] - dy], fill="black")
            im_c.save(m["crop"])
    page["fields"] = [{"field": k, "value": v} for k, v in fields]

    # 5. verification verdict
    verdict = (f"Signature: {'✔ found (' + str(len(sigs)) + ')' if sigs else '✘ not found'}\n"
               f"Stamp/seal: {'✔ found (' + str(len(stamps)) + ')' if stamps else '✘ not found'}\n"
               f"Status: {'COMPLETE - signed and stamped' if sigs and stamps else 'INCOMPLETE - check manually'}")
    page["verification"] = {"signed": bool(sigs), "stamped": bool(stamps)}

    # 6. annotated image
    im = Image.open(path).convert("RGB")
    d = ImageDraw.Draw(im)
    for b in page["blocks"]:
        d.rectangle(b["bbox_xyxy"], outline=BLOCK_COLORS.get(b.get("type"), (59, 130, 246)), width=2)
    for m in page["marks"]:
        d.rectangle(m["bbox_xyxy"], outline=MARK_COLORS[m["type"]], width=6)
        d.text((m["bbox_xyxy"][0] + 4, m["bbox_xyxy"][1] + 4), m["type"].upper(), fill=MARK_COLORS[m["type"]])
    for hb in hide:
        d.rectangle(hb, fill="black")
    img_out = f"{out}/{name}_annotated.png"
    im.save(img_out)

    # 7. exports
    json_out, md_out, xlsx_out, html_out = (f"{out}/{name}.json", f"{out}/{name}.md",
                                            f"{out}/{name}.xlsx", f"{out}/{name}_report.html")
    json.dump(page, open(json_out, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
    open(md_out, "w", encoding="utf-8").write(page["markdown"])
    tables = []
    for b in page["blocks"]:
        if b.get("type") == "Table" and "<table" in b["text"]:
            try:
                tables += pd.read_html(io.StringIO(b["text"]))
            except Exception as e:
                print("table not converted:", e)
    with pd.ExcelWriter(xlsx_out) as w:
        pd.DataFrame(fields, columns=["Field", "Value"]).to_excel(w, sheet_name="Fields", index=False)
        pd.DataFrame([{k: str(m[k]) for k in ("type", "bbox_xyxy", "conf", "method")} for m in page["marks"]],
                     columns=["type", "bbox_xyxy", "conf", "method"]).to_excel(w, sheet_name="Stamps_Signatures", index=False)
        for i, t in enumerate(tables, 1):
            t.to_excel(w, sheet_name=f"Table_{i}", index=False)
    open(html_out, "w", encoding="utf-8").write(report_html(name, verdict, fields, page, img_out))

    # 8. archive
    archive_add(name, bool(sigs), bool(stamps), fields, full)

    global LAST_TIMING
    LAST_TIMING = {"parser": t1 - t0, "marks": t2 - t1, "rest": time.time() - t2}
    del im, d
    gc.collect(); torch.cuda.empty_cache()
    return (verdict, img_out, page["markdown"], pd.DataFrame(fields, columns=["Field", "Value"]),
            [m["crop"] for m in page["marks"]], [img_out, json_out, md_out, xlsx_out, html_out])

import gradio as gr
print("pipeline ready - now run A5")

In [ ]:
# G1. Build the synthetic benchmark: realistic government pages with exact ground truth
import random, re, os, json, glob, io, math, urllib.request
import numpy as np
from PIL import Image, ImageDraw, ImageFont, ImageFilter, features

SYN = "/content/synthetic"
N_PER_LANG = 10           # 10 Marathi + 10 Hindi + 10 English = 30 pages
random.seed(2026); np.random.seed(2026)
os.makedirs(SYN, exist_ok=True)
print("complex script shaping (raqm):", features.check_feature("raqm"))

def font_path(name):
    hits = glob.glob(f"/usr/share/fonts/**/{name}", recursive=True)
    if not hits: raise FileNotFoundError(name)
    return hits[0]
FP = {("dev", False): font_path("NotoSansDevanagari-Regular.ttf"), ("dev", True): font_path("NotoSansDevanagari-Bold.ttf"),
      ("lat", False): font_path("NotoSans-Regular.ttf"), ("lat", True): font_path("NotoSans-Bold.ttf")}
CURSIVE = []
for url in ["https://github.com/google/fonts/raw/main/ofl/greatvibes/GreatVibes-Regular.ttf",
            "https://github.com/google/fonts/raw/main/ofl/allura/Allura-Regular.ttf",
            "https://github.com/google/fonts/raw/main/ofl/sacramento/Sacramento-Regular.ttf"]:
    try:
        path = "/content/" + url.rsplit("/", 1)[1]
        if not os.path.exists(path): urllib.request.urlretrieve(url, path)
        ImageFont.truetype(path, 40); CURSIVE.append(path)
    except Exception as e:
        print("cursive font not available:", url, e)
print("cursive fonts:", len(CURSIVE))

_fc = {}
def F(size, bold=False, dev=False):
    k = (size, bold, dev)
    if k not in _fc:
        kw = {"layout_engine": ImageFont.Layout.RAQM} if dev and features.check_feature("raqm") else {}
        _fc[k] = ImageFont.truetype(FP[("dev" if dev else "lat", bold)], size, **kw)
    return _fc[k]

DEVCH = re.compile(r"[ऀ-ॿ‌‍]")
def runs(text):
    out, cur, mode = [], "", None
    for ch in text:
        d = mode if ch == " " else bool(DEVCH.match(ch))
        if mode is None: mode = bool(d)
        if ch != " " and d != mode:
            out.append((cur, mode)); cur, mode = ch, d
        else:
            cur += ch
    if cur: out.append((cur, mode))
    return out
def tw(text, size, bold=False):
    return sum(F(size, bold, dv).getlength(s) for s, dv in runs(text))
def put(d, x, base, text, size, bold=False, fill=(25, 25, 25)):
    for s, dv in runs(text):
        d.text((x, base), s, font=F(size, bold, dv), fill=fill, anchor="ls")
        x += F(size, bold, dv).getlength(s)
    return x

DEVDIG = str.maketrans("0123456789", "०१२३४५६७८९")
LANG = {
 "mr": dict(head=["महाराष्ट्र शासन", "महसूल व वन विभाग", "तहसील कार्यालय, {place}"],
            titles=["जाहीर सूचना", "उत्पन्नाचा दाखला", "फेरफार नोंद सूचना"],
            para="याद्वारे सर्व संबंधितांना कळविण्यात येते की खाली नमूद केलेल्या जमिनीच्या फेरफार नोंदीबाबत हरकती असल्यास त्या पंधरा दिवसांच्या आत या कार्यालयात सादर कराव्यात. मुदतीनंतर आलेल्या हरकतींचा विचार केला जाणार नाही.",
            name="अर्जदाराचे नाव", date="दिनांक", aad="आधार क्रमांक", pan="पॅन क्रमांक", phone="मोबाईल क्रमांक",
            pin="पिन कोड", fee="शुल्क", cur="रु.", land="जमीन", survey="गट नं.", email=None,
            table=["अ.क्र.", "नाव", "भूखंड क्र.", "क्षेत्र (हे.)"], desig="तहसीलदार",
            names=["रमेश पाटील", "सुनीता जाधव", "गणेश कुलकर्णी", "अनिता देशमुख", "प्रकाश शिंदे"],
            places=["नाशिक", "पुणे", "नागपूर", "सातारा"]),
 "hi": dict(head=["भारत सरकार", "राजस्व विभाग", "जिला कार्यालय, {place}"],
            titles=["सूचना", "आय प्रमाण पत्र", "भूमि अभिलेख सूचना"],
            para="सभी संबंधित व्यक्तियों को सूचित किया जाता है कि नीचे दी गई भूमि के अभिलेख में परिवर्तन के संबंध में यदि कोई आपत्ति हो तो वह पंद्रह दिनों के भीतर इस कार्यालय में प्रस्तुत करें। निर्धारित अवधि के बाद प्राप्त आपत्तियों पर विचार नहीं किया जाएगा।",
            name="आवेदक का नाम", date="दिनांक", aad="आधार संख्या", pan="पैन संख्या", phone="मोबाइल नंबर",
            pin="पिन कोड", fee="शुल्क", cur="रुपये", land="भूमि", survey="सर्वे नं.", email=None,
            table=["क्र.", "नाम", "भूखंड सं.", "क्षेत्रफल (हे.)"], desig="तहसीलदार",
            names=["राजेश कुमार", "सीमा वर्मा", "अमित शर्मा", "पूजा सिंह", "विकास यादव"],
            places=["लखनऊ", "भोपाल", "जयपुर", "पटना"]),
 "en": dict(head=["Government of Maharashtra", "Revenue and Forest Department", "Office of the Tahsildar, {place}"],
            titles=["PUBLIC NOTICE", "INCOME CERTIFICATE", "MUTATION ENTRY NOTICE"],
            para="All concerned persons are hereby informed that objections, if any, regarding the mutation entry of the land mentioned below should be submitted to this office within fifteen days. Objections received after the due period will not be considered.",
            name="Applicant Name", date="Date", aad="Aadhaar No.", pan="PAN", phone="Mobile No.",
            pin="PIN Code", fee="Fee", cur="Rs.", land="Land", survey="Survey No.", email="E-mail",
            table=["Sr.", "Name", "Plot", "Area (ha)"], desig="Tahsildar",
            names=["Ramesh Patil", "Sunita Jadhav", "Amit Sharma", "Pooja Singh", "Vikas Yadav"],
            places=["Nashik", "Pune", "Nagpur", "Satara"]),
}
SIGN_NAMES = ["R. Patil", "S. Kulkarni", "A. Deshmukh", "V. Sharma", "P. Shinde", "M. Joshi", "K. Rao"]

def rand_values(lang):
    v = {}
    v["Date"] = random.choice(["{:02d}/{:02d}/20{}", "{:02d}.{:02d}.20{}", "{:02d}-{:02d}-20{}"]).format(
        random.randint(1, 28), random.randint(1, 12), random.randint(18, 26))
    if random.random() < 0.6: v["Aadhaar"] = f"{random.randint(2,9)}{random.randint(0,999):03d} {random.randint(0,9999):04d} {random.randint(0,9999):04d}"
    if random.random() < 0.5: v["PAN"] = "".join(random.choice("ABCDEFGHJKLMNPQRSTUVWXYZ") for _ in range(5)) + f"{random.randint(0,9999):04d}" + random.choice("ABCDEFGHJKLMNPQRSTUVWXYZ")
    if random.random() < 0.7: v["Phone"] = f"{random.randint(6,9)}{random.randint(0,999999999):09d}"
    if random.random() < 0.6: v["PIN code"] = f"{random.randint(1,8)}{random.randint(0,99999):05d}"
    if random.random() < 0.6: v["Amount"] = random.choice(["500", "1,500", "2,750", "10,000", "25"])
    if random.random() < 0.7: v["Survey"] = random.choice([f"{random.randint(1,300)}", f"{random.randint(1,300)}/{random.randint(1,9)}"])
    if lang == "en" and random.random() < 0.6:
        v["Email"] = random.choice(["tahsil", "office", "revenue", "collector"]) + f"{random.randint(1,99)}@maharashtra.gov.in"
    return v

def make_signature(name):
    ink = random.choice([(20, 35, 120), (20, 35, 120), (25, 25, 25)])
    im = Image.new("RGBA", (520, 200), (0, 0, 0, 0)); d = ImageDraw.Draw(im)
    if CURSIVE and random.random() < 0.7:
        f = ImageFont.truetype(random.choice(CURSIVE), random.randint(64, 86))
        d.text((20, 110), name, font=f, fill=ink + (255,), anchor="ls")
        x1 = 20 + f.getlength(name)
        pts = [(30, 135), (x1 * 0.5, 145 + random.randint(-8, 8)), (x1 + 20, 125)]
        d.line(pts, fill=ink + (255,), width=3, joint="curve")
    else:                                   # free-hand scribble
        x, y = 30.0, 110.0
        pts = []
        for i in range(random.randint(40, 70)):
            x += random.uniform(3, 9); y += random.uniform(-14, 14); y = min(max(y, 40), 170)
            pts.append((x, y))
        d.line(pts, fill=ink + (255,), width=3, joint="curve")
        d.line([(30, 150), (x, 140)], fill=ink + (255,), width=2)
    im = im.rotate(random.uniform(-8, 8), expand=True, resample=Image.BICUBIC)
    return im.crop(im.getbbox())

def arc_text(img, text, cx, cy, r, color, size):
    f = F(size, True, False)
    total = sum(f.getlength(c) for c in text)
    ang = -math.pi / 2 - (total / r) / 2
    for c in text:
        w = f.getlength(c)
        a = ang + (w / 2) / r
        g = Image.new("RGBA", (size * 2, size * 2), (0, 0, 0, 0))
        ImageDraw.Draw(g).text((size, size), c, font=f, fill=color, anchor="mm")
        g = g.rotate(-math.degrees(a + math.pi / 2), resample=Image.BICUBIC)
        img.alpha_composite(g, (int(cx + r * math.cos(a) - size), int(cy + r * math.sin(a) - size)))
        ang += w / r

def make_stamp(lang, place, black):
    color = (35, 35, 35) if black else random.choice([(35, 70, 190), (115, 45, 165), (190, 35, 45)])
    c = color + (255,)
    if random.random() < 0.7:               # round seal
        S = 300; im = Image.new("RGBA", (S, S), (0, 0, 0, 0)); d = ImageDraw.Draw(im)
        d.ellipse([6, 6, S - 6, S - 6], outline=c, width=6); d.ellipse([50, 50, S - 50, S - 50], outline=c, width=2)
        arc_text(im, "OFFICE OF THE TAHSILDAR", S / 2, S / 2, 122, c, 22)
        star = [(S/2 + 22 * math.cos(math.pi/2 + k*math.pi/5) * (1 if k % 2 == 0 else 0.45),
                 S/2 - 22 * math.sin(math.pi/2 + k*math.pi/5) * (1 if k % 2 == 0 else 0.45)) for k in range(10)]
        d.polygon(star, fill=c)
        lines = [LANG[lang]["desig"], place]
        for i, t in enumerate(lines):
            put(d, S/2 - tw(t, 24, True)/2, S/2 + 52 + i * 30, t, 24, True, c)
    else:                                   # rectangular stamp
        W_, H_ = 360, 170; im = Image.new("RGBA", (W_, H_), (0, 0, 0, 0)); d = ImageDraw.Draw(im)
        d.rectangle([4, 4, W_ - 4, H_ - 4], outline=c, width=5); d.rectangle([14, 14, W_ - 14, H_ - 14], outline=c, width=2)
        for i, t in enumerate([LANG[lang]["desig"], place, "SEAL"]):
            put(d, W_/2 - tw(t, 30, True)/2, 58 + i * 40, t, 30, True, c)
    a = np.array(im)                        # faded, uneven ink
    keep = (np.random.rand(*a.shape[:2]) > random.uniform(0.05, 0.25)).astype(np.float32)
    a[..., 3] = (a[..., 3] * keep * random.uniform(0.6, 0.9)).astype(np.uint8)
    im = Image.fromarray(a).rotate(random.uniform(-25, 25), expand=True, resample=Image.BICUBIC)
    return im.crop(im.getbbox())

def paste(page, im, x, y):
    page.alpha_composite(im, (int(x), int(y)))
    b = im.getchannel("A").point(lambda v: 255 if v > 40 else 0).getbbox()
    W_, H_ = page.size
    return [max(x + b[0], 0), max(y + b[1], 0), min(x + b[2], W_), min(y + b[3], H_)]

def make_page(idx, lang, scan):
    L = LANG[lang]; place = random.choice(L["places"]); name = random.choice(L["names"])
    devdig = lang == "mr" and random.random() < 0.4
    num = (lambda s: s.translate(DEVDIG)) if devdig else (lambda s: s)
    W, H = 1240, 1754
    page = Image.new("RGBA", (W, H), (252, 251, 247, 255)); d = ImageDraw.Draw(page)
    gt = {"page": f"syn_{idx:03d}", "lang": lang, "scan": scan, "devanagari_digits": devdig,
          "signatures": [], "stamps": [], "fields": [], "sensitive": []}
    y = 120
    for i, h in enumerate(L["head"]):
        t = h.format(place=place); s = [36, 30, 30][i]
        put(d, W/2 - tw(t, s, True)/2, y, t, s, True); y += int(s * 1.6)
    d.line([(90, y), (W - 90, y)], fill=(40, 40, 40), width=3); y += 80
    t = random.choice(L["titles"]); put(d, W/2 - tw(t, 40, True)/2, y, t, 40, True); y += 90
    words, line = L["para"].split(), ""
    for w in words + [None]:
        if w is None or tw(line + " " + w, 28) > W - 200:
            put(d, 100, y, line.strip(), 28); y += 48; line = ""
        if w: line += " " + w
    y += 30
    vals = rand_values(lang)
    rows = [(L["name"], name, None)]
    rows.append((L["date"], num(vals["Date"]), "Date"))
    if "Aadhaar" in vals: rows.append((L["aad"], num(vals["Aadhaar"]), "Aadhaar"))
    if "PAN" in vals: rows.append((L["pan"], vals["PAN"], "PAN"))
    if "Phone" in vals: rows.append((L["phone"], num(vals["Phone"]), "Phone"))
    if "PIN code" in vals: rows.append((L["pin"], num(vals["PIN code"]), "PIN code"))
    if "Amount" in vals: rows.append((L["fee"], f"{L['cur']} {num(vals['Amount'])}", "Amount"))
    if "Survey" in vals: rows.append((L["land"], f"{L['survey']} {num(vals['Survey'])}", "Survey / Gat no."))
    if "Email" in vals: rows.append((L["email"], vals["Email"], "Email"))
    for label, value, field in rows:
        x = put(d, 100, y, f"{label} : ", 28)
        x1 = put(d, x, y, value, 28)
        if field:
            gt["fields"].append({"field": field, "value": value})
            if field in ("Aadhaar", "PAN"):
                gt["sensitive"].append({"field": field, "value": value, "box": [x, y - 30, x1, y + 10]})
        y += 50
    if random.random() < 0.5:               # small table without field keywords
        y += 20; cols = [100, 220, 620, 860, W - 100]
        data = [L["table"]] + [[num(str(i)), random.choice(L["names"]), num(random.choice(["12", "7/1", "45/2", "118"])),
                                num(random.choice(["0.80", "1.20", "2.05"]))] for i in range(1, 4)]
        for r, row in enumerate(data):
            for ci, cell in enumerate(row):
                put(d, cols[ci] + 12, y + 40, cell, 26, r == 0)
            d.line([(cols[0], y), (cols[-1], y)], fill=(40, 40, 40), width=2); y += 56
        d.line([(cols[0], y), (cols[-1], y)], fill=(40, 40, 40), width=2)
        for cx in cols: d.line([(cx, y - 56 * len(data)), (cx, y)], fill=(40, 40, 40), width=2)
    # signature block
    by = H - 260
    put(d, W - 420, by + 120, L["desig"], 28, True); put(d, W - 420, by + 165, place, 26)
    if random.random() < 0.85:
        sig = make_signature(random.choice(SIGN_NAMES))
        gt["signatures"].append(paste(page, sig, W - 440 + random.randint(-20, 40), by + 90 - sig.height))
    if random.random() < 0.85:
        black = random.random() < 0.25
        st = make_stamp(lang, place, black)
        sx = random.choice([W - 760, W - 560, W - 420]); sy = by + random.randint(-120, 20) - st.height // 3
        gt["stamps"].append({"box": paste(page, st, sx, sy), "ink": "black" if black else "colour"})
    img = page.convert("RGB")
    if scan == "photo":                     # shadow, blur, noise, JPEG
        a = np.asarray(img).astype(np.float32)
        g = np.linspace(random.uniform(0.65, 0.8), 1.0, W)[None, :, None]
        if random.random() < 0.5: g = g[:, ::-1]
        a = a * g * np.array([1.0, 0.98, 0.92]) + np.random.normal(0, 6, a.shape)
        img = Image.fromarray(np.clip(a, 0, 255).astype(np.uint8)).filter(ImageFilter.GaussianBlur(random.uniform(0.7, 1.2)))
        buf = io.BytesIO(); img.save(buf, "JPEG", quality=55); img = Image.open(buf).convert("RGB")
        path = f"{SYN}/{gt['page']}.jpg"; img.save(path, quality=90)
    else:
        a = np.asarray(img).astype(np.float32) + np.random.normal(0, 2, (H, W, 3))
        img = Image.fromarray(np.clip(a, 0, 255).astype(np.uint8))
        path = f"{SYN}/{gt['page']}.png"; img.save(path)
    gt["path"] = path
    return gt

GT = []
for i, lang in enumerate(["mr"] * N_PER_LANG + ["hi"] * N_PER_LANG + ["en"] * N_PER_LANG):
    GT.append(make_page(i + 1, lang, "photo" if i % 2 else "scan"))
json.dump(GT, open(f"{SYN}/ground_truth.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
print(len(GT), "pages |", sum(len(g["signatures"]) for g in GT), "signatures |",
      sum(len(g["stamps"]) for g in GT), "stamps |", sum(len(g["fields"]) for g in GT), "fields")
from IPython.display import display
display(Image.open(GT[0]["path"]).resize((620, 877))); display(Image.open(GT[21]["path"]).resize((620, 877)))

In [ ]:
# E3. Run AksharDrishti on every synthetic page and score it automatically
import statistics, time
torch.cuda.reset_peak_memory_stats()
GT = json.load(open(f"{SYN}/ground_truth.json", encoding="utf-8"))
PRED, TIMES = {}, []
for i, g in enumerate(GT, 1):
    process_document(g["path"], mask=False)
    PRED[g["page"]] = json.load(open(f"/content/app_output/{g['page']}/{g['page']}.json", encoding="utf-8"))
    TIMES.append(dict(LAST_TIMING)); print(f"[{i}/{len(GT)}] {g['page']} {sum(LAST_TIMING.values()):.1f}s", end="  ")
PEAK = torch.cuda.max_memory_allocated() / 1e9

def iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy; u = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / u if u else 0.0
def match(preds, gts, thr):
    pairs = sorted(((iou(p, g), i, j) for i, p in enumerate(preds) for j, g in enumerate(gts)), reverse=True)
    up, ug, m = set(), set(), []
    for v, i, j in pairs:
        if v < thr: break
        if i not in up and j not in ug: up.add(i); ug.add(j); m.append((i, j))
    return m
def prf(tp, npred, ntrue):
    P = tp / npred if npred else float("nan"); R = tp / ntrue if ntrue else float("nan")
    F = 2*P*R/(P+R) if (P == P and R == R and P + R) else float("nan")
    return P, R, F
f2 = lambda x: "--" if x != x else f"{x:.2f}"

RES = {"pages": len(GT), "peak_gpu_gb": round(PEAK, 2)}
lines = []
def say(s=""): print(s); lines.append(s)
say("\n\n=== Dataset ===")
for k in ["mr", "hi", "en"]: say(f"{k}: {sum(g['lang']==k for g in GT)} pages")
say(f"scan/photo: {sum(g['scan']=='scan' for g in GT)}/{sum(g['scan']=='photo' for g in GT)}; "
    f"Devanagari-digit pages: {sum(g['devanagari_digits'] for g in GT)}")
say(f"signatures {sum(len(g['signatures']) for g in GT)}; stamps colour {sum(s['ink']=='colour' for g in GT for s in g['stamps'])}, "
    f"black {sum(s['ink']=='black' for g in GT for s in g['stamps'])}; fields {sum(len(g['fields']) for g in GT)}; "
    f"pages with Aadhaar/PAN {sum(bool(g['sensitive']) for g in GT)}")

for thr in (0.5, 0.3):
    say(f"\n=== Detection at IoU {thr}  (class & n & P & R & F1) ===")
    acc = {"sig": [0, 0, 0], "col": [0, 0], "blk": [0, 0], "stp_pred": 0}
    for g in GT:
        pm = PRED[g["page"]]["marks"]
        ps = [m["bbox_xyxy"] for m in pm if m["type"] == "signature"]
        pt = [m["bbox_xyxy"] for m in pm if m["type"] == "stamp"]
        acc["sig"][0] += len(match(ps, g["signatures"], thr)); acc["sig"][1] += len(ps); acc["sig"][2] += len(g["signatures"])
        mt = match(pt, [s["box"] for s in g["stamps"]], thr)
        acc["stp_pred"] += len(pt)
        for _, j in mt: acc["col" if g["stamps"][j]["ink"] == "colour" else "blk"][0] += 1
        for s in g["stamps"]: acc["col" if s["ink"] == "colour" else "blk"][1] += 1
    tp_st = acc["col"][0] + acc["blk"][0]
    rows = [("Signature", acc["sig"][2], prf(acc["sig"][0], acc["sig"][1], acc["sig"][2])),
            ("Stamp (coloured ink)", acc["col"][1], (float("nan"), acc["col"][0]/max(acc["col"][1],1), float("nan"))),
            ("Stamp (black ink)", acc["blk"][1], (float("nan"), acc["blk"][0]/max(acc["blk"][1],1), float("nan"))),
            ("Stamp (all)", acc["col"][1] + acc["blk"][1], prf(tp_st, acc["stp_pred"], acc["col"][1] + acc["blk"][1]))]
    for lbl, n, (P, R, F_) in rows: say(f"{lbl} & {n} & {f2(P)} & {f2(R)} & {f2(F_)} \\\\")
    RES[f"det_{thr}"] = {lbl: [n, P, R, F_] for lbl, n, (P, R, F_) in rows}

ok_s = sum((any(m["type"]=="signature" for m in PRED[g["page"]]["marks"])) == bool(g["signatures"]) for g in GT)
ok_t = sum((any(m["type"]=="stamp" for m in PRED[g["page"]]["marks"])) == bool(g["stamps"]) for g in GT)
tagged = sum("inside_mark" in b for g in GT for b in PRED[g["page"]]["blocks"])
say(f"\nVerdict accuracy: signed {100*ok_s/len(GT):.1f}%, stamped {100*ok_t/len(GT):.1f}%; tagged blocks {tagged}")
RES.update(verdict_signed=ok_s/len(GT), verdict_stamped=ok_t/len(GT), tagged=tagged)

def norm(v): return re.sub(r"\s", "", str(v).translate(DEV_DIGITS)).upper()
T = {(g["page"], f["field"], norm(f["value"])) for g in GT for f in g["fields"]}
Pd = {(g["page"], f["field"], norm(f["value"])) for g in GT for f in PRED[g["page"]]["fields"]}
say("\n=== Key fields (field & n & P & R & F1) ===")
RES["fields"] = {}
for fld in list(FIELD_PATTERNS):
    t_ = {x for x in T if x[1] == fld}; p_ = {x for x in Pd if x[1] == fld}
    P, R, F_ = prf(len(t_ & p_), len(p_), len(t_)); RES["fields"][fld] = [len(t_), P, R, F_]
    say(f"{fld} & {len(t_)} & {f2(P)} & {f2(R)} & {f2(F_)} \\\\")
P, R, F_ = prf(len(T & Pd), len(Pd), len(T)); RES["fields"]["micro"] = [len(T), P, R, F_]
say(f"Micro-average & {len(T)} & {f2(P)} & {f2(R)} & {f2(F_)} \\\\")
for grp, key in [("lang", "mr"), ("lang", "hi"), ("lang", "en"), ("scan", "scan"), ("scan", "photo")]:
    pg = {g["page"] for g in GT if g[grp] == key}
    t_ = {x for x in T if x[0] in pg}; p_ = {x for x in Pd if x[0] in pg}
    P, R, F_ = prf(len(t_ & p_), len(p_), len(t_)); RES["fields"][f"{grp}_{key}"] = [len(t_), P, R, F_]
    say(f"  fields on {key}: P {f2(P)} R {f2(R)} F1 {f2(F_)}")

say("\n=== Masking ===")
n_s = txt_ok = img_ok = 0
for g in GT:
    page = PRED[g["page"]]
    alltext = "\n".join([b.get("text") or "" for b in page["blocks"]] + [page.get("markdown", "")])
    masked = norm(mask_text(alltext)); boxes = sensitive_boxes(page)
    for s in g["sensitive"]:
        n_s += 1; txt_ok += norm(s["value"]) not in masked
        x0, y0, x1, y1 = map(int, s["box"]); cov = np.zeros((y1 - y0, x1 - x0), bool)
        for b in boxes:
            a0, b0 = max(int(b[0]) - x0, 0), max(int(b[1]) - y0, 0); a1, b1 = min(int(b[2]) - x0, x1 - x0), min(int(b[3]) - y0, y1 - y0)
            if a1 > a0 and b1 > b0: cov[b0:b1, a0:a1] = True
        img_ok += int(cov.mean() >= 0.9)
say(f"Text: {txt_ok}/{n_s} identity numbers hidden ({100*txt_ok/max(n_s,1):.1f}%)")
say(f"Image: {img_ok}/{n_s} identity numbers >=90% covered ({100*img_ok/max(n_s,1):.1f}%)")
RES.update(mask_n=n_s, mask_text=txt_ok, mask_img=img_ok)

say("\n=== Time per page (stage & mean & median) ===")
for lbl, k in [("Parser (layout + recognition)", "parser"), ("Signature + stamp detection", "marks"), ("Fields, masking, outputs, archive", "rest")]:
    v = [t[k] for t in TIMES]; say(f"{lbl} & {statistics.mean(v):.2f} & {statistics.median(v):.2f} \\\\"); RES[f"time_{k}"] = [statistics.mean(v), statistics.median(v)]
tot = [sum(t.values()) for t in TIMES]; say(f"Total & {statistics.mean(tot):.2f} & {statistics.median(tot):.2f} \\\\")
RES["time_total"] = [statistics.mean(tot), statistics.median(tot)]
say(f"Peak GPU memory: {PEAK:.2f} GB")
open(f"{SYN}/results_for_paper.txt", "w", encoding="utf-8").write("\n".join(lines))
print("\nRESULTS_JSON=" + json.dumps(RES, default=lambda x: x.item() if hasattr(x, "item") else str(x)))
for pg in ["syn_001", "syn_022"]:
    display(Image.open(f"/content/app_output/{pg}/{pg}_annotated.png").resize((620, 877)))